# Runtime Summary

This notebook writes one compact tracking table to `simulations/logs/runtime_simple_by_config.csv`.

`time_hms` is the observed joblib batch wall-clock time parsed from the logs. `jbl_sum_time_hms`, when present, is the cumulative per-run elapsed time stored in semiparametric `.jbl` files, so it is worker/run time rather than cluster wall time.

In [17]:
from pathlib import Path
import re

import joblib
import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for cand in [start] + list(start.parents):
        if (cand / 'simulations').exists() and (cand / 'nnpiv').exists():
            return cand
    raise RuntimeError('Could not locate repo root containing simulations/ and nnpiv/.')


REPO_ROOT = find_repo_root(Path.cwd())
LOG_DIR = REPO_ROOT / 'simulations' / 'logs'
SP_DIR = REPO_ROOT / 'simulations' / 'semiparametric_cov'
OUTPUT_CSV = LOG_DIR / 'runtime_summary.csv'

JOBS = [
    (14576905, 'sapphire', 'config_np_nn'),
    (14467784, 'test', 'config_sp_benchmark2'),
    (14467786, 'test', 'config_sp_benchmark'),
    (14469973, 'test', 'config_np_benchmark'),
    (14548827, 'test', 'config_np_benchmark2'),
    (14706007, 'test', 'config_np_benchmark2'),
    (15252595, 'sapphire', 'config_sp_nn'),
    (17834497, 'test', 'config_sp_approxrkhs2iv'),
    (17834571, 'test', 'config_sp_approxrkhs2ivl2'),
    (17834664, 'test', 'config_np_approxrkhs2iv'),
    (17834692, 'test', 'config_np_approxrkhs2ivl2'),
    (17834888, 'sapphire', 'config_sp_rkhs2ivl2'),
    (17834914, 'sapphire', 'config_sp_rkhs2iv'),
    (17835073, 'test', 'config_np_rkhs2iv'),
    (17835499, 'shared', 'config_np_rkhs2ivl2'),
    (18447460, 'test', 'config_sp_rkhs2iv'),
]


def unique_join(values, sep='/'):
    out = []
    for value in values:
        value = str(value)
        if value and value not in out:
            out.append(value)
    return sep.join(out)


def fmt_hms(seconds):
    if seconds is None or seconds == '' or pd.isna(seconds):
        return ''
    seconds = int(round(float(seconds)))
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    secs = seconds % 60
    return f'{hours}:{minutes:02d}:{secs:02d}'


def unit_seconds(value, unit):
    value = float(value)
    if unit.startswith('s'):
        return value
    if unit.startswith('min'):
        return value * 60
    if unit.startswith('h'):
        return value * 3600
    raise ValueError(f'Unknown elapsed unit: {unit}')


def config_order():
    order = []
    for _, _, config in JOBS:
        if config not in order:
            order.append(config)
    return order


def expected_batches(config):
    return 8 if config == 'config_sp_benchmark2' else 4


def method_for_config(config):
    method = config.replace('config_np_', '').replace('config_sp_', '')
    if method == 'benchmark':
        return '2SLS'
    if method == 'benchmark2':
        return '2SLS+Reg2SLS'
    if method == 'nn':
        return 'AGMM2L2' if config.startswith('config_np_') else 'AGMM2'
    return {
        'approxrkhs2iv': 'ApproxRKHS2IV',
        'approxrkhs2ivl2': 'ApproxRKHS2IVL2',
        'rkhs2iv': 'RKHS2IV',
        'rkhs2ivl2': 'RKHS2IVL2',
    }.get(method, method)


In [18]:
# Parse completed joblib batches from Slurm stderr logs.
DONE_RE = re.compile(r'Done\s+5000\s+out\s+of\s+5000\s+\|\s+elapsed:\s+([0-9.]+)([a-z]+)\s+finished')

log_rows = []
for config in config_order():
    cfg_jobs = [(job_id, partition) for job_id, partition, cfg in JOBS if cfg == config]
    elapsed_seconds = []

    for job_id, _ in cfg_jobs:
        err_path = LOG_DIR / f'myerrors_{job_id}.err'
        if not err_path.exists():
            continue
        for line in err_path.open(errors='ignore'):
            match = DONE_RE.search(line)
            if match:
                elapsed_seconds.append(unit_seconds(match.group(1), match.group(2)))

    observed = len(elapsed_seconds)
    expected = expected_batches(config)
    log_rows.append({
        'config': config,
        'partitions': unique_join([partition for _, partition in cfg_jobs]),
        'method': method_for_config(config),
        'job_ids': unique_join([job_id for job_id, _ in cfg_jobs], sep='+'),
        'completed_batches': f'{observed}/{expected}',
        'time_hms': fmt_hms(sum(elapsed_seconds)) if elapsed_seconds else '',
    })

runtime_df = pd.DataFrame(log_rows)


In [19]:
# Add cumulative per-run elapsed time from semiparametric JBL files.
JBL_RE = re.compile(
    r'^results_nexperiments_\d+_seed_\d+_(?P<profile>.+)_\d+_fn_\d+_(?P<method>[^.]+)\.jbl$'
)


def config_from_jbl(path: Path):
    match = JBL_RE.match(path.name)
    if not match:
        return None
    profile = match.group('profile')
    if profile.startswith('skipfailedruns_True_'):
        profile = profile.removeprefix('skipfailedruns_True_')
    return f'config_sp_{profile}'


def elapsed_sum_from_jbl(path: Path):
    total = 0.0
    for run in joblib.load(path):
        if not isinstance(run, (tuple, list)) or len(run) <= 3:
            continue
        try:
            value = float(run[3])
        except Exception:
            continue
        if np.isfinite(value):
            total += value
    return total


jbl_seconds_by_config = {}
for path in sorted(SP_DIR.glob('results_*_fn_*_*.jbl')):
    config = config_from_jbl(path)
    if config is None:
        continue
    jbl_seconds_by_config[config] = jbl_seconds_by_config.get(config, 0.0) + elapsed_sum_from_jbl(path)

runtime_df['jbl_sum_time_hms'] = runtime_df['config'].map(
    lambda config: fmt_hms(jbl_seconds_by_config[config]) if config in jbl_seconds_by_config else ''
)


In [20]:
LOG_DIR.mkdir(parents=True, exist_ok=True)
runtime_df.to_csv(OUTPUT_CSV, index=False)
print('Wrote', OUTPUT_CSV.relative_to(REPO_ROOT))
runtime_df


Wrote simulations/logs/runtime_summary.csv


,config,partitions,method,job_ids,completed_batches,time_hms,jbl_sum_time_hms
0,config_np_nn,sapphire,AGMM2L2,14576905,4/4,4:23:24,
1,config_sp_benchmark2,test,2SLS+Reg2SLS,14467784,8/8,0:35:22,63:11:31
2,config_sp_benchmark,test,2SLS,14467786,4/4,0:00:18,0:14:23
3,config_np_benchmark,test,2SLS,14469973,4/4,0:00:10,
4,config_np_benchmark2,test,2SLS+Reg2SLS,14548827+14706007,4/4,17:55:12,
5,config_sp_nn,sapphire,AGMM2,15252595,4/4,28:29:30,3100:36:11
6,config_sp_approxrkhs2iv,test,ApproxRKHS2IV,17834497,4/4,7:11:36,786:19:37
7,config_sp_approxrkhs2ivl2,test,ApproxRKHS2IVL2,17834571,4/4,6:39:42,728:26:07
8,config_np_approxrkhs2iv,test,ApproxRKHS2IV,17834664,4/4,2:35:18,
9,config_np_approxrkhs2ivl2,test,ApproxRKHS2IVL2,17834692,4/4,1:23:42,
